# ShopSphere Global: Data Cleaning and Preprocessing Pipeline
**Course:** IT2226 AI Innovation and Deep Learning Project  
**Dataset:** ShopSphere Global Data Pack (Part 1 Sessions & Part 2 Tables)  
**Historical Cutoff Date:** 2024-09-30  

---
## Table of Contents
1. [Overview & Objectives](#1-overview--objectives)
2. [Environment Setup & Configuration](#2-environment-setup--configuration)
3. [Step 1: Data Profiling & Auditing](#step-1-data-profiling--auditing)
4. [Step 2: Universal Boolean & Flag Normalization](#step-2-universal-boolean--flag-normalization)
5. [Step 3: Currency & Numeric String Parsing](#step-3-currency--numeric-string-parsing)
6. [Step 4: Categorical & Entity Harmonization](#step-4-categorical--entity-harmonization)
7. [Step 5: Datetime Parsing & Cutoff Date Verification](#step-5-datetime-parsing--cutoff-date-verification)
8. [Step 6: Synthetic Noise Column Removal](#step-6-synthetic-noise-column-removal)
9. [Step 7: Meaningful Structural Nulls vs. Imputable Missingness](#step-7-meaningful-structural-nulls-vs-imputable-missingness)
10. [Step 8: Table-by-Table Cleaning Implementation](#step-8-table-by-table-cleaning-implementation)
11. [Step 9: Validation, Integrity Checks & Export](#step-9-validation-integrity-checks--export)

## 1. Overview & Objectives

According to the project documentation (`DATA_README.md` and `IT2226_ShopSphere_Brief_Student.docx`):
- The dataset contains synthetic enterprise records up to **September 30, 2024**.
- Target in `train.csv`: `target = 1` if the customer makes at least one purchase between October 1 and October 30, 2024.
- The data intentionally exhibits real-world problems: inconsistent spellings, mixed boolean encodings, numbers formatted as currency strings, missing values, outliers, and system noise fields.

This notebook walks through the step-by-step cleaning methodology to prepare clean, consistent datasets ready for deep learning feature engineering.

## 2. Environment Setup & Configuration

In [ ]:
import os
import numpy as np
import pandas as pd

# Configure display options
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 1000)

# File paths
PATH_PART1 = 'ShopSphere_DataPack_Part1_sessions'
PATH_PART2 = 'ShopSphere_DataPack_Part2_other_tables'
CLEANED_OUTPUT_DIR = 'cleaned_data'
os.makedirs(CLEANED_OUTPUT_DIR, exist_ok=True)

CUTOFF_DATE = pd.Timestamp('2024-09-30')
print('Configured paths. Cutoff date set to:', CUTOFF_DATE.strftime('%Y-%m-%d'))

## Step 1: Data Profiling & Auditing
We inspect the raw data to directly observe the known issues before modifying any values.

In [ ]:
# Load raw samples for auditing
raw_customers = pd.read_csv(os.path.join(PATH_PART2, 'customers.csv'))
raw_transactions = pd.read_csv(os.path.join(PATH_PART2, 'transactions.csv'))
raw_sessions_sample = pd.read_csv(os.path.join(PATH_PART1, 'sessions.csv'), nrows=100000)

print('1. Raw country_code variants in customers:')
print(raw_customers['country_code'].dropna().unique()[:12])

print('\n2. Raw credit_limit samples (currency as string):')
print(raw_customers['credit_limit'].dropna().unique()[:5])

print('\n3. Raw age samples (includes "unknown"):')
print(raw_customers['age'].unique()[:10])

print('\n4. Raw discount_applied values in transactions (mixed booleans):')
print(raw_transactions['discount_applied'].dropna().unique())

print('\n5. Raw bounce_flag values in sessions:')
print(raw_sessions_sample['bounce_flag'].dropna().unique())

print('\n6. Raw traffic_source variants in sessions:')
print(raw_sessions_sample['traffic_source'].dropna().unique()[:8])

## Step 2: Universal Boolean & Flag Normalization
Across the tables, flags like `discount_applied`, `high_value_flag`, `bounce_flag`, `email_opt_in`, etc., use `'1'/'0'`, `'True'/'False'`, `'Yes'/'No'`, and `'Y'/'N'` interchangeably.
We map all variations to a unified numeric binary flag `(0 / 1)`.

In [ ]:
BOOLEAN_MAP = {
    '1': 1, 1: 1, 1.0: 1,
    '0': 0, 0: 0, 0.0: 0,
    'true': 1, 't': 1, True: 1,
    'false': 0, 'f': 0, False: 0,
    'yes': 1, 'y': 1,
    'no': 0, 'n': 0
}

def normalize_boolean_series(series: pd.Series) -> pd.Series:
    """Converts any common boolean representation into binary 1 or 0."""
    cleaned = series.astype(str).str.strip().str.lower().map(BOOLEAN_MAP)
    return cleaned.astype('Int8')

# Test normalization
test_sample = pd.Series(['Yes', 'no', 'TRUE', 'False', '1', '0', 'Y', 'N'])
print('Normalized sample:')
print(normalize_boolean_series(test_sample).tolist())

## Step 3: Currency & Numeric String Parsing
Monetary amounts such as `customers.credit_limit`, `transactions.order_value`, and `marketing_campaigns.campaign_budget` are formatted with dollar signs (`$`) and comma separators.
Additionally, string values like `"unknown"` in `customers.age` need to be coerced to `NaN` and cast to numeric.

In [ ]:
def parse_currency_series(series: pd.Series) -> pd.Series:
    """Strips currency formatting ($ and ,) and converts to float64."""
    return (
        series.astype(str)
        .str.replace(r'[\$,]', '', regex=True)
        .str.strip()
        .replace({'nan': np.nan, 'None': np.nan, '': np.nan})
        .astype(float)
    )

def parse_age_series(series: pd.Series) -> pd.Series:
    """Parses age, converting 'unknown' and invalid strings to NaN."""
    return pd.to_numeric(series.replace('unknown', np.nan), errors='coerce')

# Quick test
print('Cleaned credit_limit head:')
print(parse_currency_series(raw_customers['credit_limit']).head())
print('Parsed age head:')
print(parse_age_series(raw_customers['age']).head())

## Step 4: Categorical & Entity Harmonization
- Country codes are normalized to standard ISO alpha-2 (`US`, `UK`, `DE`, `FR`, `IN`).
- Casing for categories (e.g. `traffic_source`, `channel`, `device_type`) is converted to lowercase with whitespace stripped.

In [ ]:
COUNTRY_CODE_MAP = {
    'us': 'US', 'usa': 'US', 'united states': 'US', 'u.s.': 'US', 'u.s.a.': 'US',
    'uk': 'UK', 'united kingdom': 'UK', 'gb': 'UK', 'great britain': 'UK',
    'de': 'DE', 'germany': 'DE', 'deutschland': 'DE',
    'fr': 'FR', 'france': 'FR',
    'in': 'IN', 'india': 'IN'
}

def harmonize_country_codes(series: pd.Series) -> pd.Series:
    """Maps messy country names to clean ISO alpha-2 codes."""
    return series.astype(str).str.strip().str.lower().map(COUNTRY_CODE_MAP).fillna(series.str.upper().str.strip())

def harmonize_category(series: pd.Series) -> pd.Series:
    """Trims and converts text categories to lowercase."""
    return series.astype(str).str.strip().str.lower().replace({'nan': np.nan, 'none': np.nan})

print('Harmonized country sample:')
print(harmonize_country_codes(raw_customers['country_code']).value_counts())

## Step 5: Datetime Parsing & Cutoff Date Verification
All temporal fields are converted into pandas `datetime64[ns]` objects. We also assert that historical records don't contain timestamps beyond **2024-09-30**.

In [ ]:
def parse_datetimes(df: pd.DataFrame, date_columns: list) -> pd.DataFrame:
    """Parses specified date columns into datetime format."""
    df = df.copy()
    for col in date_columns:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors='coerce')
    return df

# Verify cutoff on transactions
trans_dates = pd.to_datetime(raw_transactions['transaction_timestamp'], errors='coerce')
future_trans = (trans_dates > CUTOFF_DATE).sum()
print(f'Transactions timestamp range: {trans_dates.min().strftime("%Y-%m-%d")} to {trans_dates.max().strftime("%Y-%m-%d")}')
print(f'Transactions strictly after cutoff (leakage check): {future_trans}')

## Step 6: Synthetic Noise Column Removal
`DATA_README.md` notes that `noise_customer_0..4` in `customers.csv` and `noise_trans_0..1` in `transactions.csv` are randomly generated synthetic columns with no business value.
Dropping them reduces memory usage and prevents deep learning models from overfitting on noise.

In [ ]:
NOISE_COLUMNS = [
    'noise_customer_0', 'noise_customer_1', 'noise_customer_2', 'noise_customer_3', 'noise_customer_4',
    'noise_trans_0', 'noise_trans_1'
]

def drop_noise_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Drops synthetic noise columns if present."""
    to_drop = [c for c in NOISE_COLUMNS if c in df.columns]
    return df.drop(columns=to_drop)

## Step 7: Meaningful Structural Nulls vs. Imputable Missingness
We handle missing data according to its business meaning:
1. **Structural / Valid Nulls (Do NOT Impute)**:
   - `sessions.campaign_id` $\rightarrow$ `"no_campaign"` (organic/direct sessions)
   - `deliveries.delivered_date` $\rightarrow$ Remains `NaT` for packages *In Transit* or *Lost*
   - `support_tickets.transaction_id` $\rightarrow$ `NaT` / blank for non-order general support tickets
   - `support_tickets.resolution_hours` $\rightarrow$ `NaN` for tickets that were still *Open* at cutoff
2. **Feature Missingness Indicators**:
   - `customers.age` $\rightarrow$ Add `age_is_missing` flag (1 if missing, 0 otherwise), then impute median based on the training set.

## Step 8: Table-by-Table Cleaning Implementation
We now define modular cleaning functions for each core table.

In [ ]:
def clean_customers_table(path: str) -> pd.DataFrame:
    """Clean customers table."""
    df = pd.read_csv(path)
    df = drop_noise_columns(df)
    df['country_code'] = harmonize_country_codes(df['country_code'])
    df['credit_limit'] = parse_currency_series(df['credit_limit'])
    df['age'] = parse_age_series(df['age'])
    df['age_is_missing'] = df['age'].isna().astype(int)
    df['signup_date'] = pd.to_datetime(df['signup_date'], errors='coerce')
    df['customer_tier'] = df['customer_tier'].astype(str).str.strip().str.capitalize()
    return df

def clean_transactions_table(path: str) -> pd.DataFrame:
    """Clean transactions table."""
    df = pd.read_csv(path)
    df = drop_noise_columns(df)
    df['order_value'] = parse_currency_series(df['order_value'])
    df['discount_applied'] = normalize_boolean_series(df['discount_applied'])
    df['high_value_flag'] = normalize_boolean_series(df['high_value_flag'])
    df['transaction_timestamp'] = pd.to_datetime(df['transaction_timestamp'], errors='coerce')
    df['shipping_speed'] = harmonize_category(df['shipping_speed'])
    df['payment_method'] = harmonize_category(df['payment_method'])
    return df

def clean_deliveries_table(path: str) -> pd.DataFrame:
    """Clean deliveries table."""
    df = pd.read_csv(path)
    for col in ['ship_date', 'promised_date', 'delivered_date']:
        df[col] = pd.to_datetime(df[col], errors='coerce')
    df['delivery_status'] = df['delivery_status'].astype(str).str.strip().str.title()
    df['carrier'] = df['carrier'].astype(str).str.strip()
    df['shipping_speed'] = harmonize_category(df['shipping_speed'])
    return df

def clean_support_tickets_table(path: str) -> pd.DataFrame:
    """Clean support tickets table."""
    df = pd.read_csv(path)
    df['created_date'] = pd.to_datetime(df['created_date'], errors='coerce')
    df['channel'] = harmonize_category(df['channel'])
    df['category'] = harmonize_category(df['category'])
    df['priority'] = df['priority'].astype(str).str.strip().str.capitalize()
    df['status'] = df['status'].astype(str).str.strip().str.capitalize()
    return df

def clean_customer_preferences_table(path: str) -> pd.DataFrame:
    """Clean customer preferences table."""
    df = pd.read_csv(path)
    for flag in ['email_opt_in', 'sms_opt_in', 'do_not_contact']:
        df[flag] = normalize_boolean_series(df[flag])
    df['last_updated'] = pd.to_datetime(df['last_updated'], errors='coerce')
    df['preferred_channel'] = harmonize_category(df['preferred_channel'])
    df['preferred_language'] = harmonize_category(df['preferred_language'])
    return df

def clean_marketing_campaigns_table(path: str) -> pd.DataFrame:
    """Clean marketing campaigns table."""
    df = pd.read_csv(path)
    df['campaign_budget'] = parse_currency_series(df['campaign_budget'])
    df['start_date'] = pd.to_datetime(df['start_date'], errors='coerce')
    df['end_date'] = pd.to_datetime(df['end_date'], errors='coerce')
    df['campaign_type'] = harmonize_category(df['campaign_type'])
    return df

## Step 9: Validation, Integrity Checks & Export
We execute the cleaning pipeline, verify the shapes and types, and save the standardized tables to `cleaned_data/` for modeling.

In [ ]:
print('Executing data cleaning pipeline...')

# Clean & Save Customers
cust_clean = clean_customers_table(os.path.join(PATH_PART2, 'customers.csv'))
cust_clean.to_csv(os.path.join(CLEANED_OUTPUT_DIR, 'customers_clean.csv'), index=False)
print(f'Cleaned customers: {len(cust_clean):,} rows | Cols: {len(cust_clean.columns)}')

# Clean & Save Transactions
trans_clean = clean_transactions_table(os.path.join(PATH_PART2, 'transactions.csv'))
trans_clean.to_csv(os.path.join(CLEANED_OUTPUT_DIR, 'transactions_clean.csv'), index=False)
print(f'Cleaned transactions: {len(trans_clean):,} rows | Cols: {len(trans_clean.columns)}')

# Clean & Save Deliveries
deliv_clean = clean_deliveries_table(os.path.join(PATH_PART2, 'deliveries.csv'))
deliv_clean.to_csv(os.path.join(CLEANED_OUTPUT_DIR, 'deliveries_clean.csv'), index=False)
print(f'Cleaned deliveries: {len(deliv_clean):,} rows | Cols: {len(deliv_clean.columns)}')

# Clean & Save Preferences
pref_clean = clean_customer_preferences_table(os.path.join(PATH_PART2, 'customer_preferences.csv'))
pref_clean.to_csv(os.path.join(CLEANED_OUTPUT_DIR, 'customer_preferences_clean.csv'), index=False)
print(f'Cleaned preferences: {len(pref_clean):,} rows | Cols: {len(pref_clean.columns)}')

# Clean & Save Support Tickets
tickets_clean = clean_support_tickets_table(os.path.join(PATH_PART2, 'support_tickets.csv'))
tickets_clean.to_csv(os.path.join(CLEANED_OUTPUT_DIR, 'support_tickets_clean.csv'), index=False)
print(f'Cleaned tickets: {len(tickets_clean):,} rows | Cols: {len(tickets_clean.columns)}')

# Clean & Save Marketing Campaigns
mkt_clean = clean_marketing_campaigns_table(os.path.join(PATH_PART2, 'marketing_campaigns.csv'))
mkt_clean.to_csv(os.path.join(CLEANED_OUTPUT_DIR, 'marketing_campaigns_clean.csv'), index=False)
print(f'Cleaned marketing campaigns: {len(mkt_clean):,} rows | Cols: {len(mkt_clean.columns)}')

print('\nAll cleaned tables successfully validated and exported to cleaned_data/ directory!')